In [79]:
from pathlib import Path

# This file lives in: Drought_recovery/Code/Python/Spatial_CNN
PROJECT_ROOT = Path.cwd().parents[1]

print(PROJECT_ROOT)

c:\Users\Arthur.Martin\OneDrive - Department of Health and Social Care\Documents\LSE\DV495_DISSERTATION\Drought_recovery


What this section does
This block defines the project root directory programmatically, mirroring the behaviour of an RStudio Project. The notebook lives in Drought_recovery/Code/Python, so stepping two directory levels upwards reliably points to the repository root (Drought_recovery).
All subsequent file paths are constructed relative to this root, ensuring that:

the notebook can be run from any machine,
no hard‑coded absolute paths are required,
Python and R scripts share the same project‑level directory structure.

This establishes a stable, reproducible file‑path convention for the entire Python workflow.

In [80]:
import numpy as np
import rasterio
from glob import glob

This function provides a basic mechanism for loading and stacking spatial covariates (Zi channels) into a single three‑dimensional array of shape (H, W, C).
Each raster is read as:

a single‑band image,
cast to float32 for numerical stability,
stacked along the channel dimension.

The function also extracts raster metadata from the first file, which is later used for alignment and output.

In [81]:
def load_zi_stack(zi_dir):
    zi_files = sorted(zi_dir.glob("*.tif"))

    if len(zi_files) == 0:
        raise FileNotFoundError(
            f"No .tif files found in {zi_dir}"
        )

    arrays = []
    meta = None

    for f in zi_files:
        with rasterio.open(f) as src:
            arrays.append(src.read(1).astype("float32"))

            if meta is None:
                meta = src.meta

    X = np.stack(arrays, axis=-1)
    return X, zi_files, meta

This diagnostic block verifies which raster files are present in the Zi directory and confirms that Python can correctly locate them relative to the project root.
Printing filenames explicitly helps:

detect unintended intermediate or misaligned rasters,
ensure naming conventions are respected (e.g. zi_*.tif),
catch path errors early before stacking.

In [82]:
ZI_DIR = PROJECT_ROOT / "Data" / "Data_Output" / "Zi"
print("Zi directory:", ZI_DIR)
print("Files found:")
for f in ZI_DIR.glob("*.tif"):
    print("  ", f.name)

Zi directory: c:\Users\Arthur.Martin\OneDrive - Department of Health and Social Care\Documents\LSE\DV495_DISSERTATION\Drought_recovery\Data\Data_Output\Zi
Files found:
   zi_dem_m.tif
   zi_dist_market_km.tif
   zi_dist_market_scaled.tif
   zi_slope_deg.tif
   zi_surface_water_pct.tif
   zi_surface_water_scaled.tif


This function constructs a boolean spatial mask identifying grid cells where all Zi channels are observed (i.e. non‑missing).
The resulting mask:

encodes the effective geometry of Nigeria on the raster grid,
excludes ocean, neighbouring countries, and unobserved areas,
is later used to restrict both normalisation and patch extraction.

This ensures that representation learning operates only on valid national territory rather than on padded background areas.

In [83]:
def get_valid_mask(X, min_channels_frac=0.5):
    C = X.shape[-1]
    finite_counts = np.sum(np.isfinite(X), axis=-1)
    return finite_counts >= (min_channels_frac * C)

This function standardises each Zi channel using global statistics computed only over valid cells.
Key design choices:

Normalisation is not performed patch‑wise, preserving absolute geographic meaning.
Statistics are computed once and stored, enabling reproducibility.
Missing values are filled with zeros after standardisation to prevent numerical issues during CNN training.

This step ensures that all channels are on comparable scales without introducing artificial local contrast.

In [84]:
def standardise_channels(X, mask):
    """
    Standardise each Zi channel using global stats
    computed only over valid cells
    """
    X_std = X.copy()

    C = X.shape[-1]
    stats = {}

    for c in range(C):
        vals = X[..., c][mask]
        mu = vals.mean()
        sd = vals.std()

        X_std[..., c] = (X[..., c] - mu) / sd
        stats[c] = {"mean": mu, "std": sd}

    X_std[~np.isfinite(X_std)] = 0.0  # safe fill after standardisation

    return X_std, stats

This function extracts overlapping spatial patches from the Zi tensor, which serve as training samples for the convolutional autoencoder.
Important features:

Only patches fully contained within valid territory are retained.
Patch size controls the spatial context captured (here ≈320 km × 320 km).
Stride determines the degree of spatial overlap and smoothness.
Patch centre indices are stored to later map latent embeddings back to grid cells.

This design avoids boundary artefacts and ensures that the CNN learns internal geographic structure rather than raster edges.

In [85]:
def sample_patches_from_centres(
    X,
    mask,
    patch_size=12,
    n_samples=3000,
    min_valid_frac=0.4,
    seed=42
):
    rng = np.random.default_rng(seed)

    H, W, C = X.shape
    ps = patch_size
    half = ps // 2

    valid_centres = np.argwhere(mask)

    patches = []
    centres = []

    min_valid = ps * ps * min_valid_frac

    for _ in range(n_samples):
        i, j = valid_centres[rng.integers(len(valid_centres))]

        if i - half < 0 or j - half < 0:
            continue
        if i + half > H or j + half > W:
            continue

        patch_mask = mask[i-half:i+half, j-half:j+half]
        if patch_mask.sum() < min_valid:
            continue

        patch = X[i-half:i+half, j-half:j+half, :].copy()
        patch[~patch_mask] = 0.0

        patches.append(patch)
        centres.append((i, j))

    return np.stack(patches), centres

This class wraps the extracted patches in a PyTorch Dataset, making them compatible with standard data loaders and training loops.
Key points:

Patch arrays are converted from NumPy to PyTorch tensors.
Dimension order is changed from (N, H, W, C) to (N, C, H, W) as required by convolutional layers.
No labels are included, reflecting the unsupervised nature of the autoencoder.

In [86]:
import torch
from torch.utils.data import Dataset

class ZiPatchDataset(Dataset):
    def __init__(self, patches):
        self.X = torch.from_numpy(patches).permute(0, 3, 1, 2)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx]

This helper function resamples any Zi raster to a fixed reference grid using bilinear interpolation.
It resolves minor inconsistencies arising from:

different raster origins,
off‑by‑one grid extents,
resolution rounding.

This alignment is critical to ensure that each Zi channel corresponds to the same physical location.

In [87]:
from rasterio.warp import reproject, Resampling
import numpy as np
import rasterio

def read_and_align(path, ref_meta):
    with rasterio.open(path) as src:
        src_data = src.read(1).astype("float32")

        dst = np.empty(
            (ref_meta["height"], ref_meta["width"]),
            dtype="float32"
        )

        reproject(
            source=src_data,
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_meta["transform"],
            dst_crs=ref_meta["crs"],
            resampling=Resampling.bilinear
        )

    return dst

This function constructs the final Zi tensor by:

selecting a reference raster grid,
explicitly resampling all remaining Zi layers to that grid,
stacking aligned channels into a single array.

This ensures pixel‑perfect alignment across all covariates and prevents silent spatial inconsistencies.

In [88]:
def load_zi_stack(zi_dir):
    zi_files = sorted(zi_dir.glob("zi_*.tif"))

    if len(zi_files) == 0:
        raise FileNotFoundError("No zi_*.tif files found")

    arrays = []

    # --- reference grid ---
    with rasterio.open(zi_files[0]) as ref:
        ref_meta = ref.meta.copy()
        arrays.append(ref.read(1).astype("float32"))

    # --- align all others ---
    for f in zi_files[1:]:
        aligned = read_and_align(f, ref_meta)
        arrays.append(aligned)

    X = np.stack(arrays, axis=-1)
    return X, zi_files, ref_meta

This block executes the full Zi preprocessing pipeline:

Loads and aligns spatial covariates.
Constructs a valid‑cell mask.
Standardises channels globally.
Extracts spatial patches.
Wraps patches for PyTorch training.

The result is a clean, reproducible dataset ready for convolutional representation learning.

In [89]:
ZI_DIR = PROJECT_ROOT / "Data" / "Data_Output" / "Zi"


X, zi_files, meta = load_zi_stack(ZI_DIR)
mask = get_valid_mask(X)
X_std, stats = standardise_channels(X, mask)


patches, centres = sample_patches_from_centres(
    X_std,
    mask,
    patch_size=12,
    n_samples=3000,
    min_valid_frac=0.4
)




dataset = ZiPatchDataset(patches)

print("Zi shape:", X.shape)
print("Patches:", patches.shape)
print("Channels:", patches.shape[-1])

Zi shape: (112, 124, 6)
Patches: (2835, 12, 12, 6)
Channels: 6


In [90]:
print("Valid cells:", mask.sum())
print("Share of raster valid:", mask.mean())


Valid cells: 9079
Share of raster valid: 0.6537298387096774
